# RF-DETR vs RT-DETR — all 6 subsets

Fine-tunes both detectors on every subset of DIVA-HisDB (CB55, CS18, CS863) and
U-DIADS-TL (Latin14396, Latin2, Syr341) — **12 runs** — and reports COCO mAP on the
untouched test split of each.

The training recipe is copied verbatim from
`50_modelling/02_2stage/rf_detr_vs_rt_detr_cb55.ipynb`: same image size, epochs, effective
batch, LR schedule, seed, checkpoint-selection metric and evaluation. That keeps the new
numbers directly comparable with the CB55 results already in
`80_models/02_2stage/diva-hisdb/detection/hf_detr_cb55/` (RT-DETR test mAP@50 0.6055,
RF-DETR 0.3491).

## Layout

Code comes from GitHub; data and results live on Drive:

    MyDrive/Thesis/00_data/     <- same structure as the repo's 00_data
    MyDrive/Thesis/80_models/   <- results land here, same structure as the repo

`00_data`, `80_models` and `99_evaluation` are recreated as symlinks at the repo root, so
every relative path in the repo resolves untouched.

**Data is copied from Drive to the local disk before training.** Drive's FUSE mount costs
~50-100 ms per file open, and `datasets.with_transform` re-reads every image each epoch —
training straight off Drive is many times slower than the copy.

## What has to be on Drive

Per subset, the notebook needs the COCO annotations and the images:

    00_data/DIVA-HisDB/coco_dataset_<MS>/{train,val,test}.json
    00_data/DIVA-HisDB/yolo_dataset_<MS>/images/{train,val,test}/
    00_data/U-DIADS-TL/coco_dataset_<ms>/...   +  yolo_dataset_<ms>/images/...

If a U-DIADS `coco_dataset_*` is missing, it is generated on the fly from the raw
`U-DIADS-TL/<MS>/{img-<MS>,text-line-gt-<MS>}` via `71_misc/udiads_to_coco.py`, so those
raw folders should be on Drive too.

## 1. Config

In [ ]:
REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"

DRIVE_ROOT = "/content/drive/MyDrive/Thesis"   # holds 00_data/ and 80_models/
REPO_DIR   = "/content/repo"
WORK_DIR   = "/content/work"                   # local SSD copies of data + outputs

# Subsets: (family_dir, data_family_dir, coco_dir, images_dir, run_tag)
SUBSETS = [
    ("diva-hisdb", "DIVA-HisDB", "coco_dataset_CB55",       "yolo_dataset_CB55",       "cb55"),
    ("diva-hisdb", "DIVA-HisDB", "coco_dataset_CS18",       "yolo_dataset_CS18",       "cs18"),
    ("diva-hisdb", "DIVA-HisDB", "coco_dataset_CS863",      "yolo_dataset_CS863",      "cs863"),
    ("u-diads-tl", "U-DIADS-TL", "coco_dataset_latin14396", "yolo_dataset_latin14396", "latin14396"),
    ("u-diads-tl", "U-DIADS-TL", "coco_dataset_latin2",     "yolo_dataset_latin2",     "latin2"),
    ("u-diads-tl", "U-DIADS-TL", "coco_dataset_syr341",     "yolo_dataset_syr341",     "syr341"),
]

MODEL_CHECKPOINTS = {
    "RF-DETR": "Roboflow/rf-detr-medium",
    "RT-DETR": "PekingU/rtdetr_r50vd",
}

# --- training recipe: identical to rf_detr_vs_rt_detr_cb55.ipynb ---
# Do not "optimise" these for the bigger Colab GPU: raising PER_DEVICE_BATCH_SIZE changes
# the batch-norm / gradient statistics and the CB55 numbers stop being comparable.
IMAGE_SIZE = 640
NUM_EPOCHS = 50
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 0.01
SEED = 42
NUM_WORKERS = 2

import os, sys, json, shutil, subprocess, time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
DRIVE_ROOT, REPO_DIR, WORK_DIR = map(Path, (DRIVE_ROOT, REPO_DIR, WORK_DIR))
DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"
LOCAL_EVAL   = WORK_DIR / "99_evaluation"

def sh(cmd, cwd=None):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, cwd=cwd, check=True)

print("colab runtime:", IN_COLAB)

## 2. Mount Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} not found -- create it and put 00_data/ inside"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} not found"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
print("drive OK:", DRIVE_ROOT)

## 3. Clone the repo and wire up the layout

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

for d in (LOCAL_DATA, LOCAL_MODELS, LOCAL_EVAL):
    d.mkdir(parents=True, exist_ok=True)

for name, target in (("00_data", LOCAL_DATA), ("80_models", LOCAL_MODELS),
                     ("99_evaluation", LOCAL_EVAL)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(f"{link} -> {os.readlink(link)}")

## 4. Install dependencies

In [ ]:
# requirements.txt is the repo's single source of truth; lines tagged `# [colab-skip]`
# (torch, numpy, tensorflow, ...) are dropped -- the runtime already ships them built
# against its CUDA and reinstalling costs GBs.
req = REPO_DIR / "requirements.txt"
if req.exists():
    kept = [ln for ln in req.read_text().splitlines()
            if ln.strip() and not ln.lstrip().startswith("#") and "[colab-skip]" not in ln]
    filtered = Path("/tmp/requirements-colab.txt")
    filtered.write_text("\n".join(kept) + "\n")
    print("from requirements.txt:", ", ".join(k.split("==")[0] for k in kept))
    sh(f'pip -q install -r "{filtered}"')

# Extra packages this notebook needs that the repo's training code does not.
sh('pip -q install -U "transformers>=5.14.1" "datasets>=4.0" "accelerate>=1.0" '
   '"torchmetrics>=1.4" pycocotools')
sh("nvidia-smi || true")

In [ ]:
# RF-DETR support landed in recent Transformers; verify before spending GPU hours.
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)
from transformers import AutoConfig
AutoConfig.from_pretrained("Roboflow/rf-detr-medium")
AutoConfig.from_pretrained("PekingU/rtdetr_r50vd")
print("both checkpoints resolve")

## 5. Copy the data from Drive to the local disk

Only what the 12 runs actually read: the COCO jsons and the `yolo_dataset_*/images` trees
(~450 MB total). Raw U-DIADS folders come along only when a `coco_dataset_*` has to be
generated.

In [ ]:
def copy_tree(src: Path, dst: Path) -> bool:
    if not src.exists():
        return False
    if dst.exists():
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst)
    return True

t0 = time.time()
missing_coco = []
for family_dir, data_family, coco_dir, images_dir, tag in SUBSETS:
    ok_img = copy_tree(DRIVE_DATA / data_family / images_dir / "images",
                       LOCAL_DATA / data_family / images_dir / "images")
    assert ok_img, f"missing on Drive: 00_data/{data_family}/{images_dir}/images"

    if copy_tree(DRIVE_DATA / data_family / coco_dir, LOCAL_DATA / data_family / coco_dir):
        n = len(list((LOCAL_DATA / data_family / coco_dir).glob("*.json")))
        print(f"{tag:11} images + {coco_dir} ({n} json)")
    else:
        missing_coco.append((data_family, coco_dir, tag))
        print(f"{tag:11} images OK, {coco_dir} MISSING -> will generate")

print(f"copied in {time.time() - t0:.0f}s")
print("to generate:", [t for _, _, t in missing_coco] or "none")

In [ ]:
# Generate any absent U-DIADS COCO set from the raw text-line masks. DIVA sets are not
# generated here -- diva_hisdb_to_coco.py needs the PAGE-XML tree, and all three already
# exist in the repo's data.
for data_family, coco_dir, tag in missing_coco:
    assert data_family == "U-DIADS-TL", f"cannot generate {coco_dir} automatically"
    ms = {"latin14396": "Latin14396", "latin2": "Latin2", "syr341": "Syr341"}[tag]
    ok = copy_tree(DRIVE_DATA / data_family / ms, LOCAL_DATA / data_family / ms)
    assert ok, f"need raw 00_data/{data_family}/{ms} on Drive to build {coco_dir}"
    sh(f'{sys.executable} udiads_to_coco.py --manuscript {ms}', cwd=str(REPO_DIR / "71_misc"))

for _, data_family, coco_dir, _, _ in SUBSETS:
    p = LOCAL_DATA / data_family / coco_dir
    assert p.is_dir(), f"still missing: {p}"
print("all 6 COCO sets present")

## 6. Data loading — COCO → HF `Dataset` (verbatim from the CB55 notebook)

In [ ]:
import gc
from collections import defaultdict
from dataclasses import dataclass
from functools import partial

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, Image as HFImage
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from transformers import (
    AutoImageProcessor, AutoModelForObjectDetection, Trainer, TrainingArguments, set_seed,
)
from transformers.image_transforms import center_to_corners_format

BF16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
FP16 = bool(torch.cuda.is_available() and not BF16)
set_seed(SEED)
print(f"device={'cuda' if torch.cuda.is_available() else 'cpu'} bf16={BF16} fp16={FP16} "
      f"effective batch={PER_DEVICE_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
assert torch.cuda.is_available(), "select a GPU runtime -- these detectors are unusable on CPU"


def load_coco_split(coco_root: Path, image_root: Path, split: str,
                    category_id_to_label: dict) -> Dataset:
    payload = json.loads((coco_root / f"{split}.json").read_text(encoding="utf-8"))
    annotations_by_image = defaultdict(list)
    for annotation in payload["annotations"]:
        annotations_by_image[annotation["image_id"]].append(annotation)

    records = []
    for image_info in sorted(payload["images"], key=lambda item: item["id"]):
        width, height = int(image_info["width"]), int(image_info["height"])
        image_path = image_root / split / image_info["file_name"]
        if not image_path.is_file():
            raise FileNotFoundError(image_path)

        boxes, labels, areas, annotation_ids = [], [], [], []
        for annotation in annotations_by_image[image_info["id"]]:
            x, y, box_width, box_height = map(float, annotation["bbox"])
            x = min(max(x, 0.0), float(width))
            y = min(max(y, 0.0), float(height))
            box_width = min(box_width, width - x)
            box_height = min(box_height, height - y)
            if box_width <= 0 or box_height <= 0:
                continue
            boxes.append([x, y, box_width, box_height])
            labels.append(category_id_to_label[int(annotation["category_id"])])
            areas.append(box_width * box_height)
            annotation_ids.append(int(annotation["id"]))

        if not boxes:
            raise ValueError(f"{image_path} has no valid boxes")
        records.append({
            "image_id": int(image_info["id"]), "image": str(image_path),
            "width": width, "height": height,
            "objects": {"id": annotation_ids, "bbox": boxes,
                        "category": labels, "area": areas},
        })
    return Dataset.from_list(records).cast_column("image", HFImage())


def build_subset(coco_root: Path, image_root: Path):
    payload = json.loads((coco_root / "train.json").read_text(encoding="utf-8"))
    categories = sorted(payload["categories"], key=lambda item: item["id"])
    category_id_to_label = {int(c["id"]): i for i, c in enumerate(categories)}
    id2label = {i: c["name"] for i, c in enumerate(categories)}
    label2id = {name: i for i, name in id2label.items()}
    raw = DatasetDict({
        split: load_coco_split(coco_root, image_root, split, category_id_to_label)
        for split in ("train", "val", "test")
    })
    return raw, id2label, label2id

In [ ]:
def format_annotations_as_coco(image_id, categories, areas, boxes):
    return {
        "image_id": image_id,
        "annotations": [
            {"image_id": image_id, "category_id": category, "iscrowd": 0,
             "area": area, "bbox": list(box)}
            for category, area, box in zip(categories, areas, boxes)
        ],
    }


def transform_batch(examples, image_processor):
    images, annotations = [], []
    for image_id, image, objects in zip(examples["image_id"], examples["image"], examples["objects"]):
        images.append(np.array(image.convert("RGB"), copy=True))
        annotations.append(format_annotations_as_coco(
            image_id, objects["category"], objects["area"], objects["bbox"]))
    # Keep pixel_mask: aspect-ratio padding is excluded from both models' attention.
    return image_processor(images=images, annotations=annotations, return_tensors="pt")


def make_image_processor(checkpoint: str):
    return AutoImageProcessor.from_pretrained(
        checkpoint,
        do_resize=True, size={"max_height": IMAGE_SIZE, "max_width": IMAGE_SIZE},
        do_pad=True, pad_size={"height": IMAGE_SIZE, "width": IMAGE_SIZE},
    )


def collate_fn(batch):
    result = {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": [item["labels"] for item in batch],
    }
    if "pixel_mask" in batch[0]:
        result["pixel_mask"] = torch.stack([item["pixel_mask"] for item in batch])
    return result


@dataclass
class DetectionOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor


def original_size(target):
    values = np.atleast_1d(np.asarray(target["orig_size"])).flatten()
    if len(values) < 2:
        raise ValueError(f"Invalid orig_size: {target['orig_size']}")
    return int(values[0]), int(values[1])


def normalized_cxcywh_to_absolute_xyxy(boxes, image_size):
    height, width = image_size
    boxes = center_to_corners_format(boxes)
    return boxes * torch.tensor([[width, height, width, height]], dtype=boxes.dtype)


@torch.no_grad()
def compute_metrics(evaluation_results, image_processor):
    predictions, targets = evaluation_results.predictions, evaluation_results.label_ids
    image_sizes, processed_targets, processed_predictions = [], [], []

    for target_batch in targets:
        batch_sizes = []
        for target in target_batch:
            size = original_size(target)
            batch_sizes.append(size)
            processed_targets.append({
                "boxes": normalized_cxcywh_to_absolute_xyxy(torch.as_tensor(target["boxes"]), size),
                "labels": torch.as_tensor(target["class_labels"], dtype=torch.int64),
            })
        image_sizes.append(torch.tensor(batch_sizes))

    # eval_do_concat_batches=False -> one tuple per batch; positions 1/2 are logits/boxes.
    for prediction_batch, target_sizes in zip(predictions, image_sizes):
        outputs = DetectionOutput(
            logits=torch.as_tensor(prediction_batch[1]),
            pred_boxes=torch.as_tensor(prediction_batch[2]),
        )
        processed_predictions.extend(image_processor.post_process_object_detection(
            outputs, threshold=0.0, target_sizes=target_sizes))

    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=False)
    metric.update(processed_predictions, processed_targets)
    return {key: round(value.item(), 6) for key, value in metric.compute().items()}

## 7. Result sync

Only `best_model/` and `metrics_summary.json` go to Drive. Intermediate Trainer
checkpoints stay local: with `save_total_limit=2` they are ~350 MB per run, and 12 runs
would not fit a normal Drive quota. Runs whose `metrics_summary.json` is already on Drive
are skipped, so a disconnect costs at most one run.

In [ ]:
def drive_run_dir(family_dir: str, tag: str, run_name: str) -> Path:
    return DRIVE_MODELS / "02_2stage" / family_dir / "detection" / f"hf_detr_{tag}" / run_name

def local_run_dir(family_dir: str, tag: str, run_name: str) -> Path:
    return LOCAL_MODELS / "02_2stage" / family_dir / "detection" / f"hf_detr_{tag}" / run_name

def push_run(family_dir: str, tag: str, run_name: str):
    src, dst = local_run_dir(family_dir, tag, run_name), drive_run_dir(family_dir, tag, run_name)
    dst.mkdir(parents=True, exist_ok=True)
    if (src / "best_model").is_dir():
        if (dst / "best_model").exists():
            shutil.rmtree(dst / "best_model")
        shutil.copytree(src / "best_model", dst / "best_model")
    for name in ("metrics_summary.json",):
        if (src / name).is_file():
            shutil.copy2(src / name, dst / name)
    print(f"  -> pushed to {dst}")

def already_done(family_dir: str, tag: str, run_name: str) -> bool:
    return (drive_run_dir(family_dir, tag, run_name) / "metrics_summary.json").is_file()

def load_existing_results():
    rows = []
    for family_dir, data_family, coco_dir, images_dir, tag in SUBSETS:
        for model_name in MODEL_CHECKPOINTS:
            run_name = model_name.lower().replace("-", "_")
            f = drive_run_dir(family_dir, tag, run_name) / "metrics_summary.json"
            if f.is_file():
                rows.append(json.loads(f.read_text())["summary"])
    return rows

print(f"{len(load_existing_results())}/12 runs already on Drive")

## 8. Train — 2 models × 6 subsets

In [ ]:
def train_one(model_name: str, checkpoint: str, family_dir: str, data_family: str,
              coco_dir: str, images_dir: str, tag: str):
    set_seed(SEED)   # identical stochastic ordering for both models on every subset
    run_name = model_name.lower().replace("-", "_")
    output_dir = local_run_dir(family_dir, tag, run_name)

    coco_root = LOCAL_DATA / data_family / coco_dir
    image_root = LOCAL_DATA / data_family / images_dir / "images"
    raw_datasets, id2label, label2id = build_subset(coco_root, image_root)

    image_processor = make_image_processor(checkpoint)
    transform = partial(transform_batch, image_processor=image_processor)
    datasets = DatasetDict({s: d.with_transform(transform) for s, d in raw_datasets.items()})

    model = AutoModelForObjectDetection.from_pretrained(
        checkpoint, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        run_name=f"{tag}-{run_name}",
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        max_grad_norm=MAX_GRAD_NORM,
        bf16=BF16, fp16=FP16,
        dataloader_num_workers=NUM_WORKERS,
        eval_strategy="epoch", save_strategy="epoch", logging_strategy="epoch",
        metric_for_best_model="eval_map", greater_is_better=True,
        load_best_model_at_end=True, save_total_limit=2,
        remove_unused_columns=False, eval_do_concat_batches=False,
        report_to="none", seed=SEED, data_seed=SEED,
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=datasets["train"], eval_dataset=datasets["val"],
        processing_class=image_processor, data_collator=collate_fn,
        compute_metrics=partial(compute_metrics, image_processor=image_processor),
    )

    train_result = trainer.train()
    trainer.save_model(str(output_dir / "best_model"))
    image_processor.save_pretrained(str(output_dir / "best_model"))
    validation_metrics = trainer.evaluate(datasets["val"], metric_key_prefix="validation")
    test_metrics = trainer.evaluate(datasets["test"], metric_key_prefix="test")

    record = {
        "model": model_name, "checkpoint": checkpoint,
        "family": family_dir, "subset": tag,
        "validation_mAP@50": validation_metrics["validation_map_50"],
        "validation_mAP@50-95": validation_metrics["validation_map"],
        "test_mAP@50": test_metrics["test_map_50"],
        "test_mAP@50-95": test_metrics["test_map"],
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "train_runtime_seconds": train_result.metrics.get("train_runtime"),
    }
    (output_dir / "metrics_summary.json").write_text(
        json.dumps({"summary": record, "validation": validation_metrics, "test": test_metrics},
                   indent=2), encoding="utf-8")

    del trainer, model, datasets, raw_datasets, image_processor
    gc.collect()
    torch.cuda.empty_cache()
    return record

In [ ]:
results = load_existing_results()
total = len(SUBSETS) * len(MODEL_CHECKPOINTS)

for family_dir, data_family, coco_dir, images_dir, tag in SUBSETS:
    for model_name, checkpoint in MODEL_CHECKPOINTS.items():
        run_name = model_name.lower().replace("-", "_")
        if already_done(family_dir, tag, run_name):
            print(f"[skip] {tag}/{run_name} already on Drive")
            continue
        print(f"\n{'=' * 80}\n{tag}  {model_name}  ({checkpoint})\n{'=' * 80}")
        record = train_one(model_name, checkpoint, family_dir, data_family,
                           coco_dir, images_dir, tag)
        push_run(family_dir, tag, run_name)
        results.append(record)
        print(f"  test mAP@50={record['test_mAP@50']:.4f}  "
              f"mAP@50-95={record['test_mAP@50-95']:.4f}  "
              f"({record['train_runtime_seconds']:.0f}s)   [{len(results)}/{total}]")

## 9. Report

In [ ]:
report = (
    pd.DataFrame(results)
    .pivot_table(index=["family", "subset"], columns="model",
                 values=["test_mAP@50", "test_mAP@50-95"])
    .round(4)
)
display(report)

flat = pd.DataFrame(results)[
    ["family", "subset", "model", "test_mAP@50", "test_mAP@50-95",
     "validation_mAP@50", "validation_mAP@50-95", "train_runtime_seconds"]
].sort_values(["family", "subset", "model"])
display(flat.style.format({c: "{:.4f}" for c in flat.columns if c.startswith(("test", "validation"))}))

# Metrics belong under 99_evaluation, next to every other evaluation artefact.
out = DRIVE_ROOT / "99_evaluation" / "02_2stage" / "rf_detr_vs_rt_detr_all_subsets.csv"
out.parent.mkdir(parents=True, exist_ok=True)
flat.to_csv(out, index=False)
print("saved ->", out)